In [2]:
!pip install yfinance
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from ib_insync import IB, Stock, MarketOrder, util
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import yfinance as yf
import time
import warnings
import nest_asyncio
warnings.filterwarnings('ignore')

# Fix for Jupyter/existing event loop
nest_asyncio.apply()

class IBMLTradingBot:
    def __init__(self, ticker, initial_capital=10000, host='------', port=7497, client_id='-----', use_yahoo=True):
        """
        Initialize the trading bot
        
        Args:
            ticker: Stock symbol to trade
            initial_capital: Starting capital for tracking
            host: IB Gateway/TWS host (default localhost)
            port: Paper trading port (7497 for paper, 7496 for live)
            client_id: Unique client identifier
            use_yahoo: Use Yahoo Finance for data (recommended: True)
        """
        self.ticker = ticker
        self.initial_capital = initial_capital
        self.host = host
        self.port = port
        self.client_id = client_id
        self.use_yahoo = use_yahoo
        self.ib = IB()
        self.model = None
        self.scaler = StandardScaler()
        self.is_trained = False
        self.position = 0
        self.entry_price = None
        
    def connect(self):
        """Connect to Interactive Brokers"""
        try:
            self.ib.connect(self.host, self.port, clientId=self.client_id)
            print(f"✓ Connected to Interactive Brokers on port {self.port}")
            print("✓ Paper Trading Account Active")
            
            # Get account summary
            account_values = self.ib.accountSummary()
            for item in account_values:
                if item.tag == 'NetLiquidation':
                    print(f"✓ Account Value: ${float(item.value):,.2f}")
            
            return True
        except Exception as e:
            print(f"✗ Connection failed: {e}")
            print("\nMake sure:")
            print("1. TWS or IB Gateway is running")
            print("2. API connections are enabled in TWS settings")
            print("3. Port 7497 is set for paper trading")
            return False
    
    def disconnect(self):
        """Disconnect from Interactive Brokers"""
        self.ib.disconnect()
        print("Disconnected from Interactive Brokers")
    
    def get_live_data(self, duration='1 Y', bar_size='1 day'):
        """
        Fetch live historical data (Yahoo Finance or IB)
        
        Args:
            duration: How far back to fetch (e.g., '1 Y', '6 M', '90 D')
            bar_size: Bar size (e.g., '1 day', '1 hour', '5 mins')
        """
        print(f"Fetching live data for {self.ticker}...")
        
        if self.use_yahoo:
            # Use Yahoo Finance (recommended - free and reliable)
            period_map = {'1 Y': '1y', '6 M': '6mo', '3 M': '3mo', '1 M': '1mo'}
            period = period_map.get(duration, '1y')
            
            df = yf.download(self.ticker, period=period, interval='1d', progress=False)
            df.columns = df.columns.get_level_values(0)  # Flatten multi-index
            self.data = df[['Open', 'High', 'Low', 'Close', 'Volume']]
            print(f"✓ Fetched {len(self.data)} bars from Yahoo Finance")
        else:
            # Use Interactive Brokers (requires market data subscription)
            contract = Stock(self.ticker, 'SMART', 'USD')
            self.ib.qualifyContracts(contract)
            
            bars = self.ib.reqHistoricalData(
                contract,
                endDateTime='',
                durationStr=duration,
                barSizeSetting=bar_size,
                whatToShow='TRADES',
                useRTH=True,
                formatDate=1
            )
            
            df = util.df(bars)
            df.set_index('date', inplace=True)
            df.columns = ['Open', 'High', 'Low', 'Close', 'Volume', 'Average', 'BarCount']
            self.data = df[['Open', 'High', 'Low', 'Close', 'Volume']]
            print(f"✓ Fetched {len(self.data)} bars from Interactive Brokers")
        
        return self.data
    
    def get_current_price(self):
        """Get real-time current price"""
        if self.use_yahoo:
            # Use Yahoo Finance for current price (free, 15-min delayed)
            ticker = yf.Ticker(self.ticker)
            try:
                current_price = ticker.info.get('currentPrice') or ticker.info.get('regularMarketPrice')
                if current_price:
                    return current_price
            except:
                pass
            
            # Fallback: latest close from history
            print(" Using latest close price from Yahoo Finance")
            return self.data['Close'].iloc[-1]
        else:
            # Use Interactive Brokers (requires market data subscription)
            contract = Stock(self.ticker, 'SMART', 'USD')
            self.ib.qualifyContracts(contract)
            
            ticker = self.ib.reqMktData(contract, '', False, False)
            self.ib.sleep(3)
            
            if ticker.last and not np.isnan(ticker.last):
                return ticker.last
            elif ticker.close and not np.isnan(ticker.close):
                return ticker.close
            else:
                print("⚠ Using latest historical close price")
                return self.data['Close'].iloc[-1]
    
    def calculate_indicators(self):
        """Calculate technical indicators"""
        df = self.data.copy()
        
        # RSI
        delta = df['Close'].diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
        rs = gain / loss
        df['RSI'] = 100 - (100 / (1 + rs))
        
        # MACD
        exp1 = df['Close'].ewm(span=12, adjust=False).mean()
        exp2 = df['Close'].ewm(span=26, adjust=False).mean()
        df['MACD'] = exp1 - exp2
        df['Signal_Line'] = df['MACD'].ewm(span=9, adjust=False).mean()
        
        # Bollinger Bands
        df['MA20'] = df['Close'].rolling(window=20).mean()
        df['BB_std'] = df['Close'].rolling(window=20).std()
        df['BB_upper'] = df['MA20'] + (df['BB_std'] * 2)
        df['BB_lower'] = df['MA20'] - (df['BB_std'] * 2)
        
        # Moving Averages
        df['MA5'] = df['Close'].rolling(window=5).mean()
        df['MA10'] = df['Close'].rolling(window=10).mean()
        df['MA50'] = df['Close'].rolling(window=50).mean()
        
        # Volume indicators
        df['Volume_MA'] = df['Volume'].rolling(window=20).mean()
        df['Volume_Ratio'] = df['Volume'] / df['Volume_MA']
        
        # Price momentum
        df['Momentum'] = df['Close'].pct_change(periods=10)
        
        # ATR
        high_low = df['High'] - df['Low']
        high_close = np.abs(df['High'] - df['Close'].shift())
        low_close = np.abs(df['Low'] - df['Close'].shift())
        ranges = pd.concat([high_low, high_close, low_close], axis=1)
        true_range = np.max(ranges, axis=1)
        df['ATR'] = true_range.rolling(14).mean()
        
        self.data = df
        return df
    
    def train_model(self, future_period=5, profit_threshold=0.02):
        """Train the ML model on historical data"""
        df = self.data.copy()
        
        # Create labels
        df['Future_Return'] = df['Close'].shift(-future_period) / df['Close'] - 1
        df['Target'] = (df['Future_Return'] > profit_threshold).astype(int)
        
        # Prepare features
        feature_cols = ['RSI', 'MACD', 'Signal_Line', 'MA5', 'MA10', 'MA50',
                       'Volume_Ratio', 'Momentum', 'ATR', 'BB_upper', 'BB_lower']
        
        df = df.dropna()
        X = df[feature_cols]
        y = df['Target']
        
        # Use 80% for training
        split_idx = int(len(X) * 0.8)
        X_train = X[:split_idx]
        y_train = y[:split_idx]
        
        # Scale and train
        X_train_scaled = self.scaler.fit_transform(X_train)
        
        print("Training model...")
        self.model = RandomForestClassifier(
            n_estimators=100,
            max_depth=10,
            min_samples_split=20,
            random_state=42
        )
        self.model.fit(X_train_scaled, y_train)
        
        accuracy = self.model.score(X_train_scaled, y_train)
        print(f"✓ Model trained with accuracy: {accuracy:.4f}")
        
        self.is_trained = True
        return accuracy
    
    def get_signal(self):
        """Get trading signal from the model"""
        if not self.is_trained:
            print("Model not trained yet!")
            return None, 0.0
        
        # Get latest features
        df = self.data.copy()
        feature_cols = ['RSI', 'MACD', 'Signal_Line', 'MA5', 'MA10', 'MA50',
                       'Volume_Ratio', 'Momentum', 'ATR', 'BB_upper', 'BB_lower']
        
        latest = df[feature_cols].iloc[-1:].values
        latest_scaled = self.scaler.transform(latest)
        
        prediction = self.model.predict(latest_scaled)[0]
        probability = self.model.predict_proba(latest_scaled)[0][1]
        
        return prediction, probability
    
    def get_position(self):
        """Get current position from IB account"""
        positions = self.ib.positions()
        for pos in positions:
            if pos.contract.symbol == self.ticker:
                return pos.position
        return 0
    
    def place_order(self, action, quantity):
        """
        Place order with Interactive Brokers
        
        Args:
            action: 'BUY' or 'SELL'
            quantity: Number of shares
        """
        contract = Stock(self.ticker, 'SMART', 'USD')
        self.ib.qualifyContracts(contract)
        
        order = MarketOrder(action, quantity)
        trade = self.ib.placeOrder(contract, order)
        
        print(f"✓ {action} order placed for {quantity} shares of {self.ticker}")
        
        # Wait for fill
        while not trade.isDone():
            self.ib.sleep(1)
        
        if trade.orderStatus.status == 'Filled':
            print(f"✓ Order filled at ${trade.orderStatus.avgFillPrice:.2f}")
            return True
        else:
            print(f"✗ Order status: {trade.orderStatus.status}")
            return False
    
    def execute_strategy(self, probability_threshold=0.9, stop_loss_pct=0.05):
        """Execute trading strategy based on model signal"""
        current_price = self.get_current_price()
        if current_price is None or np.isnan(current_price):
            print("Could not get current price")
            return
        
        signal, probability = self.get_signal()
        current_position = self.get_position()
        
        # If we have a position but no entry price, calculate it from account
        if current_position > 0 and self.entry_price is None:
            positions = self.ib.positions()
            for pos in positions:
                if pos.contract.symbol == self.ticker:
                    self.entry_price = pos.avgCost
                    print(f"ℹ Retrieved entry price from account: ${self.entry_price:.2f}")
                    break
        
        print(f"\n{'='*60}")
        print(f"Current Price: ${current_price:.2f}")
        print(f"Signal: {'BUY' if signal == 1 else 'HOLD/SELL'} (Probability: {probability:.2%})")
        print(f"Current Position: {current_position} shares")
        if self.entry_price and current_position > 0:
            pct_change = (current_price - self.entry_price) / self.entry_price
            print(f"Position P&L: {pct_change:.2%}")
        print(f"{'='*60}\n")
        
        # Buy signal
        if signal == 1 and probability > probability_threshold and current_position == 0:
            # Calculate quantity based on available capital
            account_value = float([v.value for v in self.ib.accountSummary() 
                                  if v.tag == 'NetLiquidation'][0])
            quantity = int((account_value * 0.95) / current_price)  # Use 95% of capital
            
            if quantity > 0:
                print(f"🔵 BUY SIGNAL - Placing order for {quantity} shares")
                if self.place_order('BUY', quantity):
                    self.entry_price = current_price
        
        # Sell signal (stop loss or model says sell)
        elif current_position > 0 and self.entry_price is not None:
            pct_change = (current_price - self.entry_price) / self.entry_price
            
            if signal == 0 or pct_change < -stop_loss_pct:
                reason = "Model signal" if signal == 0 else "Stop loss triggered"
                print(f"🔴 SELL SIGNAL ({reason}) - Closing position")
                if self.place_order('SELL', abs(current_position)):
                    print(f"Return: {pct_change:.2%}")
                    self.entry_price = None  # Reset after selling
    
    def run_live_trading(self, check_interval=3600):
        """
        Run live trading loop
        
        Args:
            check_interval: Seconds between checks (default 300 = 5 minutes)
        """
        print("\n" + "="*60)
        print("STARTING LIVE TRADING BOT")
        print("="*60)
        print(f"Trading: {self.ticker}")
        print(f"Check Interval: {check_interval} seconds")
        print("Press Ctrl+C to stop")
        print("="*60 + "\n")
        
        try:
            while True:
                # Refresh data
                self.get_live_data(duration='1 Y', bar_size='1 day')
                self.calculate_indicators()
                
                # Execute strategy
                self.execute_strategy()
                
                # Wait
                print(f"Next check in {check_interval} seconds...\n")
                time.sleep(check_interval)
                
        except KeyboardInterrupt:
            print("\n\nStopping trading bot...")
            self.disconnect()

# Example usage
if __name__ == "__main__":
    # Initialize bot for paper trading with Yahoo Finance data (recommended)
    bot = IBMLTradingBot(
        ticker='TSLA',
        initial_capital=10000,
        host='---------',
        port=7497, 
        client_id='------',
        use_yahoo=True  
    )
    
    
    if bot.connect():
        
        bot.get_live_data(duration='1 Y', bar_size='1 day')
        bot.calculate_indicators()
        bot.train_model()
        
        # Option 1: Run single strategy execution
        # bot.execute_strategy()
        
        # Option 2: Run continuous live trading (uncomment to use)
        bot.run_live_trading(check_interval=3600)  # Check every 1hrs
        
        # Disconnect
        bot.disconnect()

API connection failed: TimeoutError()


✗ Connection failed: 

Make sure:
1. TWS or IB Gateway is running
2. API connections are enabled in TWS settings
3. Port 7497 is set for paper trading
